---
title: "4.6 Regression tables you can share"
---

<div class="lesson-tools">
<a href="https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/04-regression/06-regression-tables.ipynb" class="btn btn-primary">Open in Colab</a>
<a href="https://ningzhanglab.github.io/Python-for-Beginners/downloads/04-regression/06-regression-tables.ipynb" class="btn btn-outline-secondary">Download notebook</a>
</div>

**Run in your browser.** Colab needs a Google account to run and save work. If its GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. A notebook is a document containing explanatory text and runnable code cells.

**Time:** about 40 minutes, including exercises · **Before you begin:** Complete [4.5 Fixed effects and panel models](../04-regression/05-fixed-effects.ipynb) first. Each notebook starts afresh: it does not remember variables from another lesson.

## You will learn

- Build a labelled coefficient comparison from fitted model results.
- Keep sample, uncertainty settings and omitted controls visible.
- Export a machine-readable CSV and a readable HTML table with notes.

## Why it matters

A regression table is a compact account of model choices, not a replacement for them. Readers need the outcome's units, predictor transformation, analysis sample, uncertainty method and included controls. A small p-value or a row of stars cannot supply those details. We first practise arranging made-up coefficients, then compare two real cross-section specifications on the same 142-country sample. The output is a selected-coefficient table, not a display of every parameter.

## Worked example

### Prepare the analysis libraries

Run this prepared cell **before importing other analysis libraries**. It uses the versions generated from the course's single dependency file. If needed, it installs free packages inside this temporary Python runtime, not on your own computer. An internet connection is needed for installation. If it reports that a restart is required, restart the Colab session and run from the top; mixing already loaded old code with newly installed libraries can produce misleading errors.

In [ ]:
#| code-fold: true
#| code-summary: "Prepared analysis-library setup (run first)"
# Generated from requirements.txt; installs only in this Python runtime.
import importlib.metadata
from pathlib import Path
import subprocess
import sys

requested = ['statsmodels==0.15.0']
needs_install = False
for requirement in requested:
    name, version = requirement.split("==")
    try:
        needs_install = needs_install or importlib.metadata.version(name) != version
    except importlib.metadata.PackageNotFoundError:
        needs_install = True
if needs_install:
    loaded = any(name in sys.modules for name in ["numpy", "pandas", "scipy", "statsmodels", "linearmodels"])
    constraints = Path("scratch/course-analysis-constraints.txt")
    constraints.parent.mkdir(parents=True, exist_ok=True)
    constraints.write_text('# Python 3.12; tested on macOS. CI also installs this file on Ubuntu.\n# Direct dependencies and their resolved transitive dependencies are pinned.\n# Add a library only when an available lesson or a course check needs it.\nappnope==1.0.0; sys_platform == "darwin"\nasttokens==3.0.2\nattrs==26.1.0\ncomm==0.2.3\ncontourpy==1.4.0\ncycler==0.12.1\ndebugpy==1.8.22\net_xmlfile==2.0.0\nexecuting==2.3.0\nfastjsonschema==2.22.3\nfonttools==4.68.0\nipykernel==7.4.0\nipython==9.18.1\nipython_pygments_lexers==1.1.1\njedi==0.20.1\njsonschema==4.26.0\njsonschema-specifications==2025.9.1\njupyter_client==8.10.0\njupyter_core==5.9.1\nkiwisolver==1.5.1\nmatplotlib==3.11.2\nmatplotlib-inline==0.2.2\nnbclient==0.11.0\nnbformat==5.11.1\nnest-asyncio2==1.7.4\nnumpy==2.5.4\nopenpyxl==3.1.5\npackaging==26.3\npandas==3.1.1\nparso==0.8.7\npexpect==4.9.0; sys_platform != "win32"\npillow==13.0.0\nplatformdirs==4.14.5\nprompt_toolkit==3.0.53\npsutil==7.2.2\nptyprocess==0.7.0; sys_platform != "win32"\npure_eval==0.2.4\nPygments==2.21.0\npyparsing==3.3.3\npython-dateutil==4.0.0\nPyYAML==6.0.3\npyzmq==27.3.0\nreferencing==0.37.0\nrpds-py==2026.9.1\nstack-data==0.6.3\ntornado==6.6.1\ntraitlets==5.16.1\ntyping_extensions==4.16.0\ntzdata==2026.6\nwcwidth==0.9.4\n\n# Statistics and fixed-effects models, with their newly required dependencies.\nformulaic==1.2.2\ninterface_meta==2.0.1\nlinearmodels==7.1\nmypy_extensions==1.2.0\nnarwhals==2.28.0\npatsy==1.0.3\npyhdfe==0.2.1\nscipy==1.18.1\nstatsmodels==0.15.0\nwrapt==2.5.2\n', encoding='utf-8')
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--constraint", str(constraints), *requested], check=True)
    if loaded:
        raise RuntimeError("Packages installed. Restart the notebook session, then run all cells from the top so updated libraries are used together.")
print("Analysis libraries ready:", ", ".join(requested))

Look for **Analysis libraries ready**. In the pinned course environment this cell installs nothing. Colab may take a few minutes the first time. The current course environment uses Python 3.12; an older runtime that cannot install the pinned dependencies needs a current Python runtime rather than arbitrary version substitutions.

### Separate estimates and uncertainty in a table

These coefficient and standard-error values are made-up numbers for fictional models; they illustrate layout only. They are not fitted to the course data. Keep separate columns for estimates and standard errors rather than encoding meaning in colour or unexplained symbols.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
practice = pd.DataFrame({"model": ["Fictional model A", "Fictional model B"], "income_coefficient": [3.0, 2.0], "standard_error": [0.5, 0.8], "N": [100, 100]})
practice

The columns have explicit roles, but this toy is still missing outcome units, transformation, covariance assumptions and sample definition. A table can look complete while omitting information needed for interpretation.

### Prepare the real country panel

Run this prepared setup once in a fresh session. It uses the course CSV in a local checkout, or downloads its fixed, checksum-verified public source in Colab. A checksum is a fingerprint of a file's contents; a mismatch stops the download from being used. You do not need to retype the setup. It does not use your Google Drive or any private data.

In [ ]:
# Prepared setup: use the course CSV locally; fetch its pinned source in Colab.
from pathlib import Path
import hashlib
import io
from urllib.request import urlopen
import pandas as pd

data_file = Path("data/gapminder.csv")
if not data_file.exists():
    source = "https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv"
    with urlopen(source, timeout=30) as response:
        raw = response.read()
    expected = "f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78"
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError("The data download changed. Please report this to the course maintainer.")
    downloaded = pd.read_csv(io.BytesIO(raw), sep="\t", dtype=str)
    downloaded = downloaded.rename(columns={"continent": "region", "lifeExp": "life_exp", "pop": "population", "gdpPercap": "gdp_pc"})
    data_file.parent.mkdir(parents=True, exist_ok=True)
    downloaded.to_csv(data_file, index=False, lineterminator="\n")
print("Course data ready: data/gapminder.csv")

Look for **Course data ready**. Colab working files are temporary; rerun this cell after a runtime reset. The data cover 142 countries at five-year intervals from 1952 to 2007. Each row is one country in one year.

### Fit comparable models on one sample

Select complete 2007 country observations including region. Fit the simple and region-adjusted specifications using HC1 t-based approximate inference in both. The weights are equal per country. Selecting one common sample prevents a coefficient change from being silently mixed with a sample change.

In [ ]:
countries = pd.read_csv(data_file)
sample = countries.loc[countries["year"] == 2007, ["country", "region", "life_exp", "gdp_pc"]].dropna().copy()
sample["log_gdp_pc"] = np.log(sample["gdp_pc"])
model_a = smf.ols("life_exp ~ log_gdp_pc", data=sample, missing="raise").fit(cov_type="HC1", use_t=True)
model_b = smf.ols('life_exp ~ log_gdp_pc + C(region, Treatment(reference="Africa"))', data=sample, missing="raise").fit(cov_type="HC1", use_t=True)
print("N for models A and B:", int(model_a.nobs), int(model_b.nobs))

Both samples contain 142 countries. The added region indicators change the specification, not the units or year. These models compare cross-country fitted associations in 2007; the country/year fixed-effects panel model from lesson 4.5 uses a different sample and comparison and should not be labelled as another 2007 cross-section.

### Extract results without retyping numbers

The function below takes a fitted result and returns a named record for the log-income coefficient. Read the coefficient, standard error, confidence interval and sample size directly from the result. Keep full precision in the result object; rounding is for display and export.

In [ ]:
def coefficient_record(result, label, region_controls):
    interval = result.conf_int().loc["log_gdp_pc"]
    return {
        "model": label,
        "estimate_years_per_log_unit": result.params["log_gdp_pc"],
        "standard_error": result.bse["log_gdp_pc"],
        "ci95_lower": interval.iloc[0],
        "ci95_upper": interval.iloc[1],
        "n_countries": int(result.nobs),
        "year": 2007,
        "region_controls": region_controls,
        "covariance": "HC1; t-based approximate intervals",
    }

coefficient_table = pd.DataFrame([
    coefficient_record(model_a, "A: income only", "No"),
    coefficient_record(model_b, "B: income + region", "Yes; Africa reference"),
])
coefficient_table.round(3)

The income coefficients are about 7.203 and 4.631 years per natural-log unit. The table exposes the uncertainty convention and the included region controls. It reports one selected coefficient from each model; omitted region coefficients are controls, not evidence that no group differences were estimated. A p-value, if reported elsewhere, is not the probability that a causal effect is absent.

### Make a compact reading view

Round a separate reading view to three decimals and transpose selected fields so each model is a column. Friendly row labels clarify units. This display keeps the two confidence bounds as separate rows; concatenated stars and parentheses are not necessary for a readable comparison.

In [ ]:
reading_table = coefficient_table.round(3).set_index("model")[["estimate_years_per_log_unit", "standard_error", "ci95_lower", "ci95_upper", "n_countries", "region_controls"]].T
reading_table.index = ["Income coefficient (years per log unit)", "HC1 standard error", "95% interval: lower", "95% interval: upper", "Country observations", "Region indicators included"]
reading_table

Read a coefficient together with its uncertainty, not as a rank of model quality. Both models remain descriptive. Their R-squared values are omitted here to keep the focus on the specified comparison; a larger R-squared would not make one model a better causal design.

### Export results with notes

Save the structured, one-row-per-model CSV for later checking. Save a separate HTML reading table with its caption and notes. HTML is a webpage document; a browser can display or print it. The simple table uses ordinary text and headings, not colour as a carrier of statistical meaning.

In [ ]:
output_dir = Path("downloads/outputs")
output_dir.mkdir(parents=True, exist_ok=True)
notes = "Outcome: life expectancy at birth, years. Selected predictor: natural log of GDP per person in the historical extract's inflation-adjusted US dollars. Sample: 142 equally weighted countries in 2007, complete on required fields. HC1 standard errors and t-based approximate 95% intervals. Model B includes source-region indicators with Africa as reference; those coefficients are not shown in the selected-coefficient table. No causal interpretation is established. Source: Gapminder via Jennifer Bryan, CC0."
csv_file = output_dir / "regression-models.csv"
html_file = output_dir / "regression-table.html"
coefficient_table.to_csv(csv_file, index=False, float_format="%.6f")
html = '<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1"><title>Income and life expectancy: selected coefficients</title><style>body{font:16px/1.6 Georgia,serif;max-width:65rem;margin:2rem auto;padding:1rem}table{border-collapse:collapse}th,td{padding:.5rem;border:1px solid #777;text-align:left}.table-wrap{overflow-x:auto}</style></head><body><h1>Income and life expectancy: selected coefficients</h1><p>Historical country cross-section, 2007</p><div class="table-wrap">' + reading_table.to_html(escape=True) + '</div><p>' + notes + '</p></body></html>'
html_file.write_text(html, encoding="utf-8")
print(csv_file)
print(html_file)

On the website download the [structured CSV](https://ningzhanglab.github.io/Python-for-Beginners/downloads/outputs/regression-models.csv) or [reading table](https://ningzhanglab.github.io/Python-for-Beginners/downloads/outputs/regression-table.html). In Colab, download the generated files through the Files panel under downloads → outputs. The source-data CC0 terms do not select a license for the course's original text or table presentation. Check the reported values against the fitted results before placing the table in a paper.

## Try it

Work in your own notebook copy. Add a new code cell for each task, try it yourself, then open the solution to compare. Run from the top if a name is missing.

### 1. Verify the exported coefficient

Read the structured CSV and compare model A's coefficient with its fitted result, allowing for the six-decimal export rounding.

<details><summary>Solution 1</summary>

Run this complete solution after the worked example.

In [ ]:
reloaded = pd.read_csv(csv_file)
print(reloaded.loc[0, "estimate_years_per_log_unit"])
print(model_a.params["log_gdp_pc"])

The values agree up to six-decimal rounding. Reading back an export can reveal swapped rows, wrong columns or unintended precision loss; it does not independently validate the regression design.

</details>

### 2. Keep a full coefficient record

Export every coefficient and standard error of model B to another CSV. Do not label it as the selected-coefficient table.

<details><summary>Solution 2</summary>

Run this complete solution after the worked example.

In [ ]:
all_coefficients = pd.DataFrame({"term": model_b.params.index, "estimate": model_b.params.to_numpy(), "HC1_standard_error": model_b.bse.to_numpy()})
all_csv = output_dir / "regression-all-coefficients.csv"
all_coefficients.to_csv(all_csv, index=False, float_format="%.6f")
all_coefficients

The [full coefficient CSV](https://ningzhanglab.github.io/Python-for-Beginners/downloads/outputs/regression-all-coefficients.csv) includes the intercept and four region contrasts as well as log income. It needs the same sample, units and uncertainty notes as the selected-coefficient table. Africa has no separate contrast because it is the reference.

</details>

### 3. Translate the model comparison

Add a displayed column for each model's fitted difference associated with twice the income, using log(2). Why should its unit be years?

<details><summary>Solution 3</summary>

Run this complete solution after the worked example.

In [ ]:
doubling_table = coefficient_table[["model", "estimate_years_per_log_unit"]].copy()
doubling_table["fitted_doubling_difference_years"] = doubling_table["estimate_years_per_log_unit"] * np.log(2)
doubling_table.round(3)

The differences are about 4.993 and 3.210 years. These are differences in the unlogged life-expectancy outcome for an income ratio, not percentage changes in life expectancy or causal income effects.

</details>

## When it goes wrong

The following examples deliberately trigger specific errors. The prepared `try`/`except` wrapper catches that named error and prints its message so the notebook can continue. It is for these demonstrations; an unexpected error elsewhere should still stop execution. You can read the message without memorizing the wrapper.

### A result attribute is guessed

Fitted-result objects use package-specific attribute names.

In [ ]:
try:
    model_a.standard_errors
except AttributeError as error:
    print(f"{type(error).__name__}: {error}")

statsmodels uses bse for standard errors; linearmodels uses std_errors. Inspect the package API rather than copying a field name across estimators.

### A coefficient is requested by the wrong term

The formula fitted log income, not raw dollar income.

In [ ]:
try:
    model_b.params["gdp_pc"]
except KeyError as error:
    print(f"{type(error).__name__}: {error}")

Use the fitted formula term log_gdp_pc and report its transformation in the table. Merely renaming the row to “income” would obscure the coefficient's unit and interpretation.

## Recap

- Extract table values from fitted results instead of retyping them.
- Name the outcome, predictor transformation, sample, controls and uncertainty method.
- Keep selected coefficients distinct from a full parameter table.
- Verify exports and retain notes; presentation does not establish causal identification.

**Analysis path complete.** Parts 0–4 now cover Python, data handling, charts and the core regression methods. Continue to [5.1 Optional local installation](../05-workflow/01-local-installation.ipynb), or stay in the browser for the workflow examples. Review the [course map](https://ningzhanglab.github.io/Python-for-Beginners/lessons/index.html) and practise explaining the sample, units and uncertainty behind a result before moving on.